# Tahap 5 — Empirical Digital Twin / Canonical State Evaluation

Notebook ini mengevaluasi integritas representasi Canonical Twin State terhadap telemetry aktual. Fokusnya adalah schema conformity, source-to-state mapping, completeness, temporal integrity, data-quality behavior, preservasi nilai, dan determinisme—bukan forecasting. Implementasi komputasi tetap berada di `src/twin_state/`.

In [ ]:
# Environment Check
from pathlib import Path
import importlib.metadata
import os
import platform
import sys

EXPECTED_DATASET_SHA256 = 'ca7831a188a191edbf82a673fac90dbb875b5095986ed07699c02530f2a02a0e'
REPOSITORY_URL = 'https://github.com/rehanalfarizu/new_jurnal.git'
IS_COLAB = 'google.colab' in sys.modules

def find_repository_root():
    candidates = [Path.cwd(), Path.cwd() / 'new_jurnal', Path.cwd().parent]
    for candidate in candidates:
        if (candidate / 'configs' / 'experiment.yaml').is_file() and (candidate / 'src').is_dir():
            return candidate.resolve()
    return None

REPO_ROOT = find_repository_root()
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('Google Colab:', IS_COLAB)
print('Repository path:', REPO_ROOT or 'belum ditemukan')
for package in ['numpy', 'pandas', 'scikit-learn', 'matplotlib', 'PyYAML', 'ipykernel']:
    try:
        version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        version = 'belum terpasang'
    print(f'{package}: {version}')

In [ ]:
# Google Colab Setup
import subprocess

if IS_COLAB and REPO_ROOT is None:
    clone_target = Path('/content/new_jurnal')
    if not clone_target.exists():
        subprocess.run(['git', 'clone', REPOSITORY_URL, str(clone_target)], check=True)
    REPO_ROOT = clone_target.resolve()
elif REPO_ROOT is None:
    raise RuntimeError('Repository tidak ditemukan. Jalankan notebook dari root new_jurnal atau direktori notebooks/.')

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('Working directory:', Path.cwd())

In [ ]:
# Dependency Setup
import importlib.util

if IS_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
    print('Dependency dari requirements.txt telah dipasang.')
else:
    required_modules = {'pandas': 'pandas', 'PyYAML': 'yaml'}
    missing = [name for name, module in required_modules.items() if importlib.util.find_spec(module) is None]
    if missing:
        raise ModuleNotFoundError('Dependency kernel belum lengkap: ' + ', '.join(missing) + f'. Jalankan {sys.executable} -m pip install -r requirements.txt')
    print('Dependency lokal terverifikasi pada kernel aktif.')

In [ ]:
# Dataset Path dan Checksum
import hashlib
import warnings

# Opsi Google Drive (aktifkan dan sesuaikan hanya bila diperlukan):
# from google.colab import drive
# drive.mount('/content/drive')
# os.environ['SENSOR_DATA_PATH'] = '/content/drive/MyDrive/.../sensor_data.csv'

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

configured_path = os.environ.get('SENSOR_DATA_PATH', '').strip()
default_path = (REPO_ROOT / 'data/raw/sensor_data.csv').resolve()
if configured_path:
    DATASET_PATH = Path(configured_path).expanduser()
    if not DATASET_PATH.is_absolute():
        DATASET_PATH = (REPO_ROOT / DATASET_PATH).resolve()
    path_source = 'environment variable SENSOR_DATA_PATH'
elif default_path.is_file():
    DATASET_PATH, path_source = default_path, 'data/raw/sensor_data.csv'
elif not IS_COLAB:
    candidates_by_file = {}
    for pattern in ('*/Data/sensor_data.csv', '*/data/sensor_data.csv'):
        for path in REPO_ROOT.parent.glob(pattern):
            if path.is_file():
                info = path.stat()
                candidates_by_file[(info.st_dev, info.st_ino)] = path.resolve()
    candidates = sorted(candidates_by_file.values())
    if len(candidates) > 1:
        raise RuntimeError('Lebih dari satu dataset ditemukan; tetapkan SENSOR_DATA_PATH secara eksplisit.')
    DATASET_PATH = candidates[0] if candidates else default_path
    path_source = 'satu kandidat pada folder proyek saudara' if candidates else 'data/raw/sensor_data.csv'
else:
    DATASET_PATH, path_source = default_path, 'data/raw/sensor_data.csv'

if not DATASET_PATH.is_file():
    raise FileNotFoundError(f'sensor_data.csv tidak ditemukan di {DATASET_PATH}. Tetapkan SENSOR_DATA_PATH atau gunakan Google Drive.')
DATASET_SHA256 = sha256_file(DATASET_PATH)
CHECKSUM_MATCH = DATASET_SHA256 == EXPECTED_DATASET_SHA256
print('Dataset path:', DATASET_PATH)
print('Sumber resolusi path:', path_source)
print('SHA-256:', DATASET_SHA256)
print('Checksum sesuai dataset penelitian:', CHECKSUM_MATCH)
if not CHECKSUM_MATCH:
    warnings.warn('Checksum berbeda; evaluasi tidak dijalankan tanpa review eksplisit.', RuntimeWarning)
    raise ValueError('Dataset berbeda dari dataset penelitian.')

In [ ]:
# Load Canonical Configuration
import pandas as pd
import yaml
from IPython.display import display

with Path('configs/experiment.yaml').open(encoding='utf-8') as handle:
    experiment_config = yaml.safe_load(handle)
canonical_config = experiment_config['canonical_state']
display(pd.DataFrame([
    {'setting': 'schema_version', 'value': canonical_config['schema_version']},
    {'setting': 'room_id', 'value': canonical_config['room_id']},
    {'setting': 'source_timezone', 'value': experiment_config['data']['source_timezone']},
    {'setting': 'timezone_policy', 'value': experiment_config['preprocessing']['timezone_policy']},
    {'setting': 'gap_threshold_seconds', 'value': experiment_config['preprocessing']['gap_threshold_seconds']},
    {'setting': 'staleness_seconds', 'value': canonical_config['staleness_seconds']},
]))

In [ ]:
# Raw Dataset Summary
dataset_summary = pd.read_csv('results/tables/dataset_summary.csv')
summary_sha = str(dataset_summary.loc[dataset_summary['metrik'] == 'sha256', 'nilai'].iloc[0])
if summary_sha != DATASET_SHA256:
    raise AssertionError('Dataset summary Tahap 2 tidak cocok dengan dataset aktif.')
display(dataset_summary)

In [ ]:
# Canonical Transformation Evaluation
import json
from src.twin_state.evaluation import evaluate_canonical_state

MANIFEST_PATH = Path('results/metrics/canonical_state_evaluation_manifest.json')
FORCE_RERUN = os.environ.get('FORCE_CANONICAL_EVALUATION', '0') == '1'
manifest_current = False
if MANIFEST_PATH.is_file():
    existing = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
    code_current = all(Path(path).is_file() and sha256_file(path) == checksum for path, checksum in existing.get('code_sha256', {}).items())
    manifest_current = existing.get('input_sha256') == DATASET_SHA256 and existing.get('config_sha256') == sha256_file('configs/experiment.yaml') and code_current
if FORCE_RERUN or not manifest_current:
    evaluation_result = evaluate_canonical_state(input_path=DATASET_PATH)
    print('Pipeline evaluasi canonical dijalankan ulang.')
else:
    print('Output resmi yang cocok dengan dataset, konfigurasi, dan source code dimuat ulang.')
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
display(pd.DataFrame([manifest['summary']]).T.rename(columns={0: 'value'}))

In [ ]:
# Schema Conformity
evaluation = pd.read_csv('results/tables/canonical_state_evaluation.csv')
display(evaluation[evaluation['category'].isin(['transformation', 'schema', 'type', 'timestamp'])])

In [ ]:
# Source-to-State Mapping
field_mapping = pd.read_csv('results/tables/canonical_field_mapping.csv')
display(field_mapping)
if not (field_mapping['mapping_status'] == 'pass').all():
    raise AssertionError('Ditemukan mapping source-to-state yang tidak mempertahankan nilai.')

In [ ]:
# State Completeness
data_quality = pd.read_csv('results/tables/digital_twin_data_quality.csv')
display(data_quality[data_quality['category'].isin(['expected_from_telemetry', 'metadata', 'derived_quality'])])
print('room_id dan staleness_seconds tidak menurunkan completeness telemetry karena keduanya bukan field telemetry yang tersedia pada CSV.')

In [ ]:
# Temporal Integrity
display(evaluation[evaluation['category'].isin(['timestamp', 'temporal'])])
print('UTC dinormalisasi berdasarkan provenance; tidak ada klaim physical-to-digital latency.')

In [ ]:
# Data-Quality Evaluation
display(data_quality[data_quality['category'].isin(['validation_behavior', 'limitation'])])
print('Synthetic invalid cases diuji melalui unit test dan tidak dicampurkan dengan hasil dataset aktual.')

In [ ]:
# Transformation Correctness
correctness = evaluation[evaluation['category'] == 'correctness']
display(correctness)
display(field_mapping[['source_field', 'canonical_field', 'evaluated_records', 'value_mismatches', 'mapping_status']])

In [ ]:
# Determinism Checks
determinism = evaluation[evaluation['category'] == 'determinism']
display(determinism)
if int(determinism.loc[determinism['metric'] == 'determinism_failures', 'value'].iloc[0]) != 0:
    raise AssertionError('Transformasi canonical tidak deterministik.')

In [ ]:
# Results Tables
print('Tabel resmi:')
for name, path in manifest['official_outputs'].items():
    output_path = Path(path)
    print(f'- {name}: {output_path} ({"tersedia" if output_path.is_file() else "tidak tersedia"})')

In [ ]:
# Limitations
from IPython.display import Markdown
limitations = '\n'.join(f'- {item}' for item in manifest['limitations'])
display(Markdown('### Keterbatasan yang tidak dapat dievaluasi\n\n' + limitations))

In [ ]:
# Reproducibility Summary
reproducibility = {
    'dataset_sha256': manifest['input_sha256'],
    'config_sha256': manifest['config_sha256'],
    'code_sha256': manifest['code_sha256'],
    'git_commit': manifest.get('git_commit'),
    'working_tree_dirty': manifest.get('working_tree_dirty'),
    'python_version': sys.version.split()[0],
    'schema_version': manifest['schema_version'],
    'generated_at_utc': manifest['generated_at_utc'],
    'official_outputs': manifest['official_outputs'],
}
print(json.dumps(reproducibility, ensure_ascii=False, indent=2))